# 🏥 Medical Transcription & SOAP Note Generator

**Assignment:** Build a pipeline that:
1. Ingests a medical dictation audio file
2. Transcribes it verbatim using AI
3. Structures the transcript into a standardized SOAP note

**Stack:** Python · Anthropic Claude API · Flask (web demo)

---

| Component | Tool | Reason |
|-----------|------|--------|
| Speech-to-Text | **Claude claude-sonnet-4-5** | Native audio understanding, handles medical terminology accurately, no separate STT service needed |
| SOAP Structuring | **Claude claude-sonnet-4-5** | Best-in-class instruction following, reliable JSON output, strong clinical knowledge |
| Web Demo | **Flask + Docker** | Lightweight, easy to deploy, zero-dependency frontend |

## 📦 Install Dependencies

In [ ]:
!pip install anthropic python-dotenv -q

## 🔧 Configuration

In [ ]:
import os, json, base64, re
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import anthropic

load_dotenv()

# ── Config ────────────────────────────────────────────────────────────────────
AUDIO_FILE   = "audio/sample_dictation.mp3"   # ← path to the provided audio
OUTPUT_DIR   = Path("outputs")
MODEL        = "claude-sonnet-4-5"             # Claude model used for both tasks
AUDIO_MIME   = "audio/mpeg"                    # change to audio/wav for .wav files

OUTPUT_DIR.mkdir(exist_ok=True)

API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not API_KEY:
    API_KEY = input("Enter your Anthropic API key: ").strip()

client = anthropic.Anthropic(api_key=API_KEY)
print(f"✅ Ready  |  Model: {MODEL}  |  Audio: {AUDIO_FILE}")

---
## 🎙️ Part A — Audio Transcription (Speech-to-Text)

We use **Claude claude-sonnet-4-5's native audio understanding** to transcribe the dictation.

**Why this approach?**
- No separate STT service or model download required
- Claude handles medical terminology (drug names, anatomical terms, abbreviations) far better than generic STT
- Single API call, single API key — simpler architecture
- Privacy-safe: audio is encrypted in transit, not stored by Anthropic

In [ ]:
def transcribe_audio(audio_path: str, mime_type: str = "audio/mpeg") -> dict:
    """
    Transcribe a medical dictation audio file using Claude's native audio understanding.

    Args:
        audio_path : Path to the audio file (MP3, WAV, M4A, OGG, WebM)
        mime_type  : MIME type of the audio file

    Returns:
        dict with 'transcript' and metadata
    """
    print(f"🎙️  Loading audio: {audio_path}")
    with open(audio_path, "rb") as f:
        audio_b64 = base64.standard_b64encode(f.read()).decode()

    file_size_mb = Path(audio_path).stat().st_size / 1_048_576
    print(f"    File size: {file_size_mb:.1f} MB")
    print("    Sending to Claude API...")

    response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": mime_type,
                        "data": audio_b64
                    }
                },
                {
                    "type": "text",
                    "text": (
                        "Transcribe this medical dictation audio verbatim, word for word. "
                        "Preserve all medical terminology, drug names, dosages, and clinical "
                        "abbreviations exactly as spoken. "
                        "Output only the raw transcript text — no labels, no commentary, no timestamps."
                    )
                }
            ]
        }]
    )

    transcript = " ".join(
        block.text for block in response.content if block.type == "text"
    ).strip()

    result = {
        "transcript": transcript,
        "model": MODEL,
        "generated_at": datetime.now().isoformat(),
        "audio_file": audio_path,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }

    print(f"✅ Transcription complete")
    print(f"   Input tokens : {response.usage.input_tokens:,}")
    print(f"   Output tokens: {response.usage.output_tokens:,}")
    print(f"   Word count   : {len(transcript.split()):,}")
    return result


# ─── Run ──────────────────────────────────────────────────────────────────────
transcription_result = transcribe_audio(AUDIO_FILE, AUDIO_MIME)
raw_transcript = transcription_result["transcript"]

print("\n" + "="*60)
print("RAW TRANSCRIPT:")
print("="*60)
print(raw_transcript)

In [ ]:
# Save raw transcript to file
transcript_path = OUTPUT_DIR / "raw_transcript.txt"
with open(transcript_path, "w") as f:
    f.write(f"Medical Dictation Transcript\n")
    f.write(f"Generated : {transcription_result['generated_at']}\n")
    f.write(f"Model     : {MODEL}\n")
    f.write(f"Audio file: {AUDIO_FILE}\n")
    f.write("=" * 60 + "\n\n")
    f.write(raw_transcript)

print(f"✅ Saved → {transcript_path}")

---
## 📋 Part B — SOAP Note Generation (Text → Structured Note)

We use a carefully engineered prompt to make Claude structure the transcript into a valid SOAP note with strict section separation.

**Prompt engineering strategy:**
- Explicit definitions of each SOAP section with examples
- Hard rule against cross-contamination (subjective info in Objective section)
- JSON schema specified in the prompt for reliable structured output
- LLM self-validates its own output (and we also run a programmatic check)

In [ ]:
SOAP_SYSTEM_PROMPT = """
You are a clinical documentation specialist with expertise in medical transcription.
Convert raw physician dictation into a structured SOAP note.

SOAP definitions:
- S (Subjective): Patient-reported info — symptoms, complaints, pain levels,
  medications they mention, history. This is what the PATIENT says.
- O (Objective): Clinician-observed data ONLY — vital signs, physical exam findings,
  lab results, imaging. NOT patient-reported feelings or statements.
- A (Assessment): Clinician's diagnosis or clinical impression.
- P (Plan): Treatment, prescriptions, referrals, follow-up instructions.

CRITICAL BOUNDARY RULE:
  ✅ Subjective: "Patient reports pain 7/10" (patient told the doctor)
  ✅ Objective : "Tenderness on palpation" (doctor physically observed)
  ❌ NEVER put patient-reported pain ratings or feelings in Objective.
  ❌ NEVER put physical exam findings in Subjective.

Respond ONLY with valid JSON, no markdown fences, no preamble:
{
  "patient_info": {
    "age": "string or null",
    "sex": "string or null",
    "chief_complaint": "one-line summary"
  },
  "soap_note": {
    "subjective": {
      "chief_complaint": "string",
      "history_of_present_illness": "string",
      "pain_scale": "string or null",
      "current_medications": ["list"],
      "other_subjective": "string or null"
    },
    "objective": {
      "vital_signs": "string or null",
      "physical_exam": "string",
      "diagnostic_results": "string or null"
    },
    "assessment": {
      "diagnosis": "string",
      "clinical_impression": "string"
    },
    "plan": {
      "medications": ["list"],
      "procedures": ["list"],
      "follow_up": "string",
      "patient_education": "string or null"
    }
  },
  "validation": {
    "subjective_objective_boundary_check": "PASS or FAIL",
    "boundary_check_notes": "explanation"
  }
}
"""

def generate_soap_note(transcript: str) -> dict:
    """
    Generate a structured SOAP note from medical dictation transcript.

    Args:
        transcript: Raw transcribed dictation text

    Returns:
        Parsed SOAP note dict
    """
    print("🤖 Generating SOAP note with Claude...")

    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=SOAP_SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": f"Convert this medical dictation into a structured SOAP note:\n\n{transcript}"
        }]
    )

    raw = " ".join(
        block.text for block in response.content if block.type == "text"
    ).strip()

    # Strip markdown fences if model added them despite instructions
    raw = re.sub(r"^```[a-z]*\n?", "", raw)
    raw = re.sub(r"\n?```$", "", raw).strip()

    soap_data = json.loads(raw)
    print(f"✅ SOAP note generated")
    print(f"   Boundary check (LLM): {soap_data['validation']['subjective_objective_boundary_check']}")
    return soap_data


# ─── Run ──────────────────────────────────────────────────────────────────────
soap_result = generate_soap_note(raw_transcript)

print("\n" + "="*60)
print("SOAP NOTE (JSON):")
print("="*60)
print(json.dumps(soap_result, indent=2))

---
## ✅ Bonus — Programmatic S/O Boundary Validation

Defense-in-depth: we also run a rule-based check to catch any subjective markers that leaked into the Objective section.

In [ ]:
SUBJECTIVE_MARKERS = [
    "patient reports", "patient states", "patient denies",
    "he reports", "she reports", "he says", "she says",
    "rates the pain", "rates pain", "out of 10", "/10",
    "patient feels", "patient notes", "according to the patient",
]

def validate_soap_boundaries(soap_data: dict) -> dict:
    """
    Programmatic check: ensure no subjective (patient-reported) markers
    appear in the Objective section (clinician-observed only).
    """
    obj = soap_data.get("soap_note", {}).get("objective", {})
    obj_text = " ".join(str(v).lower() for v in obj.values() if v)

    issues = [m for m in SUBJECTIVE_MARKERS if m in obj_text]
    programmatic = "PASS" if not issues else "FAIL"
    llm_check = soap_data.get("validation", {}).get(
        "subjective_objective_boundary_check", "NOT_REPORTED"
    )
    overall = "PASS" if programmatic == "PASS" and llm_check == "PASS" else "REVIEW_NEEDED"

    return {
        "programmatic_check": programmatic,
        "llm_self_check": llm_check,
        "issues_found": issues,
        "overall_status": overall
    }


validation = validate_soap_boundaries(soap_result)

print("SOAP BOUNDARY VALIDATION")
print("=" * 40)
print(f"Programmatic check : {validation['programmatic_check']}")
print(f"LLM self-check     : {validation['llm_self_check']}")
print(f"Overall status     : {validation['overall_status']}")
if validation['issues_found']:
    print("\n⚠️  Issues detected:")
    for i in validation['issues_found']:
        print(f"  - {i}")
else:
    print("\n✅ No boundary violations detected")

---
## 💾 Save All Outputs

In [ ]:
def soap_to_markdown(soap_data: dict, transcript: str = "") -> str:
    """Format SOAP JSON as clean Markdown."""
    s  = soap_data["soap_note"]["subjective"]
    o  = soap_data["soap_note"]["objective"]
    a  = soap_data["soap_note"]["assessment"]
    p  = soap_data["soap_note"]["plan"]
    pi = soap_data["patient_info"]
    v  = soap_data["validation"]

    return "\n".join([
        "# SOAP Note",
        f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  ",
        f"**Model:** {MODEL}\n",
        "---\n",
        "## Patient Information",
        f"- **Age:** {pi.get('age','N/A')}",
        f"- **Sex:** {pi.get('sex','N/A')}",
        f"- **Chief Complaint:** {pi.get('chief_complaint','N/A')}\n",
        "---\n",
        "## S — Subjective *(Patient-reported)*",
        f"**CC:** {s.get('chief_complaint','N/A')}  ",
        f"**HPI:** {s.get('history_of_present_illness','N/A')}  ",
        f"**Pain Scale:** {s.get('pain_scale','N/A')}  ",
        f"**Current Medications:** {', '.join(s.get('current_medications',[]))or'None'}  ",
        (f"**Other:** {s['other_subjective']}" if s.get('other_subjective') else ""),
        "\n---\n",
        "## O — Objective *(Clinician-observed)*",
        f"**Vital Signs:** {o.get('vital_signs','Not documented')}  ",
        f"**Physical Exam:** {o.get('physical_exam','N/A')}  ",
        f"**Diagnostic Results:** {o.get('diagnostic_results','None')}  ",
        "\n---\n",
        "## A — Assessment *(Diagnosis)*",
        f"**Diagnosis:** {a.get('diagnosis','N/A')}  ",
        f"**Clinical Impression:** {a.get('clinical_impression','N/A')}  ",
        "\n---\n",
        "## P — Plan *(Treatment)*",
        f"**Medications:** {', '.join(p.get('medications',[]))or'None'}  ",
        f"**Procedures:** {', '.join(p.get('procedures',[]))or'None'}  ",
        f"**Follow-up:** {p.get('follow_up','N/A')}  ",
        (f"**Patient Education:** {p['patient_education']}" if p.get('patient_education') else ""),
        "\n---\n",
        "## ✅ Validation",
        f"**S/O Boundary Check:** {v.get('subjective_objective_boundary_check','N/A')}  ",
        f"**Notes:** {v.get('boundary_check_notes','N/A')}  ",
        ("\n---\n\n## Raw Transcript\n\n" + transcript) if transcript else ""
    ])


# ─── Save JSON ────────────────────────────────────────────────────────────────
output_bundle = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "model": MODEL,
        "audio_file": AUDIO_FILE
    },
    "raw_transcript": raw_transcript,
    "soap_note": soap_result,
    "validation": validation
}

json_path = OUTPUT_DIR / "soap_note.json"
with open(json_path, "w") as f:
    json.dump(output_bundle, f, indent=2)

md_path = OUTPUT_DIR / "soap_note.md"
with open(md_path, "w") as f:
    f.write(soap_to_markdown(soap_result, raw_transcript))

print("\nAll outputs saved:")
print(f"  📝 {OUTPUT_DIR}/raw_transcript.txt")
print(f"  📋 {OUTPUT_DIR}/soap_note.json")
print(f"  📄 {OUTPUT_DIR}/soap_note.md")

---
## 📊 Display Final SOAP Note

In [ ]:
from IPython.display import Markdown, display
display(Markdown(soap_to_markdown(soap_result)))

---
## 🚀 Run the Web Demo (Optional)

A full web interface is included. Run with Docker:

```bash
# 1. Clone the repo
git clone https://github.com/YOUR_USERNAME/medscribe-ai.git
cd medscribe-ai

# 2. Set API key
cp .env.example .env
# Edit .env and add your ANTHROPIC_API_KEY

# 3. Build and run
docker-compose up --build

# 4. Open browser
# http://localhost:8080
```

**Live Demo:** [https://medscribe-ai.onrender.com](https://medscribe-ai.onrender.com) *(see README for deployment steps)*